# 🗓 Calendar Agent — Kaggle LoRA Training

Colab 대안. Kaggle은 별도 GPU quota를 제공 (주 30h, T4×2 또는 P100 16GB).

## 실행 전 준비 (한 번만)

### 1. Kaggle 설정 (우측 패널)
- **Accelerator**: `GPU T4 x2` 또는 `GPU P100` 선택
- **Internet**: `On` (외부 패키지·HF 모델 다운로드용)
- **Persistence**: `Variables and Files` (선택 — 세션 재시작 시 일부 보존)

### 2. 데이터 업로드 (Kaggle Dataset으로)
1. https://www.kaggle.com/datasets → `New Dataset`
2. 3개 파일 업로드:
   - `train.jsonl` (로컬 `D:\calendar-agent\data\processed\train.jsonl`)
   - `val.jsonl` (동)
   - `golden.jsonl` (로컬 `D:\calendar-agent\data\eval\golden.jsonl`)
3. Dataset 이름: `calendar-messages` (또는 임의)
4. Visibility: **Private**
5. Create

### 3. 이 노트북에 Dataset 첨부
- 우측 패널 `+ Add Input` → `Datasets` 탭 → 위에서 만든 dataset 검색 → 첨부
- 첨부되면 `/kaggle/input/datasets/sooryongbyun/calendar-messages/` 경로에 파일 보임

### 4. GitHub PAT 발급 (Private repo clone용)
https://github.com/settings/tokens/new?type=classic — repo scope, 7 days

---

준비 끝나면 셀을 위에서 아래로 순서대로 실행. 예상 시간: ~1시간.

## 0. GPU 확인

`Tesla T4` 또는 `Tesla P100` 표시되어야 함. 안 보이면 우측 패널 Accelerator 다시.

In [ ]:
!nvidia-smi

## 1. 레포 clone (Private)

Kaggle은 working dir이 `/kaggle/working/`. 그 안에 clone.

In [ ]:
import os, getpass

%cd /kaggle/working
if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/soo-vibe/calendar-agent.git
    token = None
else:
    print('calendar-agent already cloned, force-syncing to origin/main…')
    !cd calendar-agent && git fetch origin && git reset --hard origin/main

%cd /kaggle/working/calendar-agent
!git log --oneline -1

## 2. 학습 의존성 설치 + Colab/Kaggle 호환 cleanup

Colab과 동일 — install + torchao 제거.

In [ ]:
!pip install -q -e .[train]
!pip uninstall torchao -y
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"                             # 단일 GPU 강제 (0.5B엔 2-GPU DataParallel이 오히려 느림)
os.environ["WANDB_DISABLED"] = "true"                               # wandb 완전 비활성 (로그인 프롬프트/끝-프리즈 방지)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # CUDA 단편화 완화 (OOM 방지)
import torch
print(f'
torch={torch.__version__}  cuda={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## 3. 데이터 복사 (Kaggle Dataset → 작업 디렉토리)

첨부한 Dataset이 `/kaggle/input/datasets/sooryongbyun/calendar-messages/`에 마운트됨.
작업 디렉토리에 repo 구조대로 복사.

**참고**: Dataset 이름이 다르면 `DATASET_DIR`만 수정.

In [ ]:
import shutil
from pathlib import Path

DATASET_DIR = Path('/kaggle/input/datasets/sooryongbyun/calendar-messages')
TRAIN = Path('data/processed/train.jsonl')
VAL   = Path('data/processed/val.jsonl')
GOLD  = Path('data/eval/golden.jsonl')
TRAIN.parent.mkdir(parents=True, exist_ok=True)
GOLD.parent.mkdir(parents=True, exist_ok=True)

# 항상 덮어쓰기: 이전 세션(Persistence)이나 repo의 구버전 파일이 남아 새 데이터셋을
# 안 가져오는 사고 방지. (이전 라운드에서 golden이 'skipping copy' 됐던 문제)
for fn, dst in [('train.jsonl', TRAIN), ('val.jsonl', VAL), ('golden.jsonl', GOLD)]:
    src = DATASET_DIR / fn
    if not src.exists():
        raise FileNotFoundError(f'{src} 없음. 데이터셋 새 버전 첨부 / 파일명 확인.')
    shutil.copy(src, dst)
    n = sum(1 for _ in open(dst, encoding='utf-8'))
    print(f'{src.name} -> {dst}  ({n} records)')


## 4. Sanity check + config 자동 조정

- 3개 파일 shape 검증
- WandB 끄기
- bf16 미지원 GPU(T4)면 fp16로 전환

In [ ]:
import orjson, torch
from pathlib import Path

for p in [Path('data/processed/train.jsonl'), Path('data/processed/val.jsonl'), Path('data/eval/golden.jsonl')]:
    assert p.exists(), f'MISSING: {p}'
    n = sum(1 for line in open(p, 'rb') if line.strip())
    print(f'{p}: {n} records')

# wandb off
!sed -i 's/report_to: wandb/report_to: none/' configs/train.yaml

# bf16 -> fp16 if needed
if not torch.cuda.is_bf16_supported():
    !sed -i 's/bf16: true/bf16: false/' configs/train.yaml
    !sed -i 's/fp16: false/fp16: true/' configs/train.yaml
    !sed -i "s|bnb_4bit_compute_dtype: bfloat16|bnb_4bit_compute_dtype: float16|" configs/train.yaml
    print('GPU에 bf16 미지원 — fp16로 전환')
else:
    print('bf16 사용 (P100/A100)')

!grep -E 'report_to|bf16|fp16|bnb_4bit_compute' configs/train.yaml

## 5. 학습 실행

약 2040건 × 3 epochs ≈ 192 step. T4/P100 기준 ~1시간.

**Kaggle 세션 한도**: 9시간/세션 — 우리 학습은 1시간이라 무관.

In [ ]:
# DDP: T4 2개를 진짜로 병렬 사용 (torchrun, DataParallel 아님)
!torchrun --nproc_per_node=2 scripts/train_lora.py --config configs/train.yaml

## 5.5 학습 직후 평가 (Kaggle GPU, 선택)

merge 후 골든셋으로 바로 평가해 `time_match_rate`/`final_score`를 확인한다. 로컬 CPU 평가(약 14분)보다 GPU가 훨씬 빠르다. 결과 json은 아래 6번 zip에 포함된다.

In [ ]:
# 라운드 자동 인식: train.yaml의 output_dir에서 이름/라운드를 읽어 경로 자동 구성
# (다음 라운드는 train.yaml만 바꾸면 됨 — 이 셀은 영구히 수정 불필요)
import yaml, os
_cfg = yaml.safe_load(open('configs/train.yaml', encoding='utf-8'))
LORA_DIR = _cfg['output_dir']                 # 예: models/lora/r7-qwen
NAME = os.path.basename(LORA_DIR)             # r7-qwen
ROUND = NAME.split('-')[0]                    # r7
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON = f'logs/eval_{NAME}.json'
ZIP = f'/kaggle/working/lora_{ROUND}.zip'
print(f'round={ROUND}  lora={LORA_DIR}  merged={MERGED_DIR}  eval={EVAL_JSON}  zip={ZIP}')

# 학습 직후 Kaggle GPU에서 골든셋 평가
!python scripts/merge_lora.py --base Qwen/Qwen2.5-0.5B-Instruct --lora {LORA_DIR} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval data/eval/golden.jsonl --out {EVAL_JSON}
print(f'--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

## 6. 결과 패키징 (Kaggle Output 자동 노출)

Kaggle은 `/kaggle/working/` 안의 파일을 노트북 우측 패널 `Output`에서 다운로드 가능.
zip을 `/kaggle/working/` 루트에 두면 됨.

In [ ]:
!ls -la {LORA_DIR}/
!zip -r {ZIP} {LORA_DIR} configs/train.yaml configs/lora.yaml configs/model_qwen.yaml {EVAL_JSON}
!ls -la {ZIP}

## 다운로드 방법

**FileLink는 Kaggle 프록시에서 동작하지 않는다(404 남).** 아래 방법을 쓸 것:

1. **우상단 `Save Version` → `Quick Save`** (재실행 안 함, 현재 `/kaggle/working` 스냅샷) → 저장되면 버전 열기 → **Output** 탭에서 `lora_<라운드>.zip` 다운로드  ← 가장 확실
2. 또는 에디터 우측 **Output(`/kaggle/working`) 파일 패널**에서 `lora_<라운드>.zip` 다운로드 (안 보이면 새로고침)
3. 또는 로컬에서 Kaggle API: `kaggle kernels output <user>/<notebook-slug> -p .`

학습 직후 평가(아래 5.5)를 돌렸다면 `logs/eval_<라운드>-qwen.json`의 `time_match_rate` / `final_score`를 셀 출력에서 바로 확인할 수 있다.

In [ ]:
# 참고: FileLink는 Kaggle에서 종종 404. 위 '다운로드 방법'의 Quick Save -> Output 사용 권장.
from IPython.display import FileLink
FileLink(ZIP)